# SatQuery: train the flood specialist on a free Kaggle GPU

Trains `flood_mapper`'s water models on **Sen1Floods11** (446 hand-labelled flood scenes from 11 countries,
including India) in three variants: radar (Sentinel-1), multispectral (Sentinel-2 / Landsat) and true colour
(photos and screenshots). Each is scored on the official test split, and on **Bolivia**, a country kept out of
training entirely, against the classic method SatQuery already uses; only variants that beat it are used.

**Settings (right-hand panel):** Accelerator **GPU T4 x1**, Internet **On**. Then **Save Version -> Save & Run All**.
Expected time: about 30-45 minutes. Output: `satquery_flood_models.zip`.

In [ ]:
# 1. Code (the generalize branch) and libraries
!nvidia-smi -L
!rm -rf /tmp/SatQuery && git clone -q --depth 1 -b generalize https://github.com/Vansh-Vaid/SatQuery.git /tmp/SatQuery
!cd /tmp/SatQuery && git log --oneline -1
!pip install -q rasterio 2>&1 | tail -1

In [ ]:
# 2. Sen1Floods11 hand-labelled data from its public Google Cloud bucket (about 3-4 GB)
import json, os, time, urllib.parse, urllib.request
from concurrent.futures import ThreadPoolExecutor

BUCKET, ROOT = "sen1floods11", "/tmp/sen1floods11"
PREFIXES = ["v1.1/data/flood_events/HandLabeled/S1Hand/", "v1.1/data/flood_events/HandLabeled/S2Hand/",
            "v1.1/data/flood_events/HandLabeled/LabelHand/", "v1.1/splits/flood_handlabeled/"]

def list_objects(prefix):
    names, token = [], None
    while True:
        q = {"prefix": prefix, "fields": "items(name),nextPageToken", "maxResults": 1000}
        if token:
            q["pageToken"] = token
        url = f"https://storage.googleapis.com/storage/v1/b/{BUCKET}/o?" + urllib.parse.urlencode(q)
        d = json.load(urllib.request.urlopen(url, timeout=60))
        names += [i["name"] for i in d.get("items", [])]
        token = d.get("nextPageToken")
        if not token:
            return names

def fetch(name):
    dst = os.path.join(ROOT, name)
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        return
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    url = f"https://storage.googleapis.com/{BUCKET}/" + urllib.parse.quote(name)
    for attempt in range(4):
        try:
            urllib.request.urlretrieve(url, dst + ".part")
            os.replace(dst + ".part", dst)
            return
        except Exception:
            time.sleep(3 * (attempt + 1))
    raise RuntimeError(f"could not download {name}")

t0 = time.time()
names = [n for p in PREFIXES for n in list_objects(p) if n.endswith((".tif", ".csv"))]
print({p.split("/")[-2]: sum(n.startswith(p) for n in names) for p in PREFIXES})
if not names:
    top = json.load(urllib.request.urlopen(f"https://storage.googleapis.com/storage/v1/b/{BUCKET}/o?delimiter=/&prefix=v1.1/", timeout=60))
    raise SystemExit(f"nothing found; top-level folders are: {top.get('prefixes')}")
with ThreadPoolExecutor(16) as ex:
    list(ex.map(fetch, names))
print(f"{len(names)} files in {(time.time() - t0) / 60:.1f} min")

In [ ]:
# 3. Train the three variants (radar, multispectral, true colour); score on the test split and on Bolivia
!cd /tmp/SatQuery && python training/train_flood.py --root /tmp/sen1floods11 --out /kaggle/working/flood_seg.pt --epochs 60

In [ ]:
# 4. Verdict per variant, and pack the results
import json, zipfile
m = json.load(open("/kaggle/working/flood_seg.pt.metrics.json"))
for v, r in m["variants"].items():
    verdict = "BETTER: the app will use it" if r["beats_baseline"] else "not better: the app keeps the classic method"
    held = f" | Bolivia (never seen) {r['bolivia_iou']:.3f} vs {r['bolivia_baseline_iou']:.3f}" if "bolivia_iou" in r else ""
    print(f"{v:4s} test IoU {r['test_iou']:.3f} vs {r['baseline']} {r['baseline_iou']:.3f}{held} -> {verdict}")
print("test countries:", ", ".join(m["test_countries"]))
with zipfile.ZipFile("/kaggle/working/satquery_flood_models.zip", "w") as z:
    z.write("/kaggle/working/flood_seg.pt", "flood_seg.pt")
    z.write("/kaggle/working/flood_seg.pt.metrics.json", "flood_seg.pt.metrics.json")
print()
print("Download satquery_flood_models.zip from the Output tab.")

## Afterwards
Unzip into the project's `checkpoints/` folder. The app picks up `flood_seg.pt` automatically and answers flood
questions with it (only the variants marked BETTER).